In [1]:
import pandas as pd

In [2]:
data = pd.read_csv("./data/task_2_data_ex.csv")
data.head()

,year,month,produced_material,produced_material_production_type,produced_material_release_type,produced_material_quantity,component_material,component_material_production_type,component_material_release_type,component_material_quantity,plant_id
0,2024,1,10000,8002,FIN,990.00,50000,8002.0,PROD,990.00,RLT_10
1,2024,1,50000,8002,PROD,859.00,80070,8007.0,PROD,879.00,RLT_10
2,2024,1,50000,8002,PROD,859.00,90000,NaN,ADD,50.00,RLT_10
3,2024,1,50000,8002,PROD,859.00,90001,NaN,ADD,20.00,RLT_10
4,2024,1,80070,8007,PROD,929.00,80010,8001.0,PROD,"3,626.00",RLT_10


In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1320 entries, 0 to 1319
Data columns (total 11 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   year                                1320 non-null   int64  
 1   month                               1320 non-null   int64  
 2   produced_material                   1320 non-null   int64  
 3   produced_material_production_type   1320 non-null   int64  
 4   produced_material_release_type      1320 non-null   object 
 5   produced_material_quantity          1320 non-null   object 
 6   component_material                  1320 non-null   int64  
 7   component_material_production_type  480 non-null    float64
 8   component_material_release_type     1320 non-null   object 
 9   component_material_quantity         1320 non-null   object 
 10  plant_id                            1320 non-null   object 
dtypes: float64(1), int64(5), object(5)
memory u

In [4]:
data.loc[data['produced_material_release_type'] == 'FIN', 'produced_material'].unique()

array([10000, 10001, 10002, 10003, 10004, 10005, 10006, 10007, 10008,
       10009])

In [5]:
def parse_financial_num(col):
    if col.dtype == 'object':
        return col.str.replace(',', '', regex=False).str.strip()
    return col

data['produced_material_quantity'] = pd.to_numeric(
    parse_financial_num(data['produced_material_quantity']), errors='coerce'
).fillna(0)
data['component_material_quantity'] = pd.to_numeric(
    parse_financial_num(data['component_material_quantity']), errors='coerce'
).fillna(0)

data = data.groupby([
    'plant_id', 'year',
    'produced_material', 'produced_material_release_type', 'produced_material_production_type',
    'component_material', 'component_material_release_type', 'component_material_production_type'
], observed=False, dropna=False).agg({
    'produced_material_quantity': 'sum',
    'component_material_quantity': 'sum'
}).reset_index()

print(f"Записей после годовой агрегации: {len(data)}")

Записей после годовой агрегации: 110


In [6]:
data

,plant_id,year,produced_material,produced_material_release_type,produced_material_production_type,component_material,component_material_release_type,component_material_production_type,produced_material_quantity,component_material_quantity
0,RLT_10,2024,10000,FIN,8002,50000,PROD,8002.0,11708.0,11708.0
1,RLT_10,2024,10001,FIN,8002,50001,PROD,8002.0,12023.0,12023.0
2,RLT_10,2024,50000,PROD,8002,80070,PROD,8007.0,9538.0,11303.0
3,RLT_10,2024,50000,PROD,8002,90000,ADD,NaN,9538.0,598.0
4,RLT_10,2024,50000,PROD,8002,90001,ADD,NaN,9538.0,242.0
...,...,...,...,...,...,...,...,...,...,...
105,RLT_16,2024,80077,PROD,8007,90037,ADD,NaN,10751.0,365.0
106,RLT_16,2024,80077,PROD,8007,90038,ADD,NaN,10751.0,122.0
107,RLT_16,2024,80078,PROD,8007,80018,PROD,8001.0,10676.0,41471.0
108,RLT_16,2024,80078,PROD,8007,90042,ADD,NaN,10676.0,355.0


In [14]:
cols_to_use = [
    'plant_id', 'year',
    'produced_material', 'produced_material_release_type',
    'produced_material_production_type', 'produced_material_quantity',
    'component_material', 'component_material_release_type',
    'component_material_production_type', 'component_material_quantity'
]

df = data[cols_to_use].copy()

first_level = df[df['produced_material_release_type'] == 'FIN'].copy()
first_level = first_level.rename(columns={
    'produced_material': 'fin_material_id',
    'produced_material_release_type': 'fin_material_release_type',
    'produced_material_production_type': 'fin_material_production_type',
    'produced_material_quantity': 'fin_production_quantity',
    'component_material': 'prod_material_id',
    'component_material_release_type': 'prod_material_release_type',
    'component_material_production_type': 'prod_material_production_type',
    'component_material_quantity': 'prod_production_quantity'
})

first_level = first_level.merge(
    df,
    left_on=['plant_id', 'year', 'prod_material_id'],
    right_on=['plant_id', 'year', 'produced_material'],
    how='left',
    suffixes=('', '_next')
)

first_level = first_level.rename(columns={
    'component_material': 'component_id',
    'component_material_release_type': 'component_material_release_type',
    'component_material_production_type': 'component_material_production_type',
    'component_material_quantity': 'component_consumption_quantity'
})

existing_cols = [
    'plant_id',
    'year',
    'fin_material_id',
    'fin_material_release_type',
    'fin_material_production_type',
    'fin_production_quantity',
    'prod_material_id',
    'prod_material_release_type',
    'prod_material_production_type',
    'prod_production_quantity',
    'component_id',
    'component_material_release_type',
    'component_material_production_type',
    'component_consumption_quantity'
 ]

first_level = first_level[existing_cols].dropna(subset=['component_id']).copy()

current_level = first_level.copy()
current_level['path'] = current_level.apply(
    lambda r: [str(r['fin_material_id']), str(r['component_id'])], axis=1
)

all_prod_levels = []

while not current_level.empty:
    save_cols = existing_cols + ['path']
    all_prod_levels.append(current_level[save_cols].copy())

    next_step = current_level.merge(
        df,
        left_on=['plant_id', 'year', 'component_id'],
        right_on=['plant_id', 'year', 'produced_material'],
        how='inner',
        suffixes=('', '_next')
    )

    if next_step.empty:
        break

    next_step = next_step.assign(
        prod_material_id=next_step['component_id'],
        prod_material_release_type=next_step['component_material_release_type'],
        prod_material_production_type=next_step['component_material_production_type'],
        prod_production_quantity=next_step['component_consumption_quantity'],
        component_id=next_step['component_material'],
        component_material_release_type=next_step['component_material_release_type_next'],
        component_material_production_type=next_step['component_material_production_type_next'],
        component_consumption_quantity=next_step['component_material_quantity']
    )

    next_step['new_id_str'] = next_step['component_id'].astype(str)
    cycle_mask = next_step.apply(lambda r: r['new_id_str'] in r['path'], axis=1)
    next_step = next_step[~cycle_mask].copy()

    if next_step.empty:
        break

    next_step['path'] = next_step.apply(lambda r: r['path'] + [r['new_id_str']], axis=1)
    current_level = next_step[save_cols].copy()

    if (current_level['component_material_release_type'] != 'PROD').all():
        all_prod_levels.append(current_level)
        break

first_level_expanded = pd.concat(all_prod_levels, ignore_index=True)
first_level_expanded['path'] = first_level_expanded['path'].apply(tuple)
first_level_expanded = first_level_expanded.drop_duplicates()

In [8]:
first_level_expanded.head()

,plant_id,year,fin_material_id,fin_material_release_type,fin_material_production_type,fin_production_quantity,prod_material_id,prod_material_release_type,prod_material_production_type,prod_production_quantity,component_id,component_material_release_type,component_material_production_type,component_consumption_quantity,path
0,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,80070,PROD,8007.0,11303.0,"(10000, 80070)"
1,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,90000,ADD,NaN,598.0,"(10000, 90000)"
2,RLT_10,2024,10000,FIN,8002,11708.0,50000,PROD,8002.0,11708.0,90001,ADD,NaN,242.0,"(10000, 90001)"
3,RLT_10,2024,10001,FIN,8002,12023.0,50001,PROD,8002.0,12023.0,80071,PROD,8007.0,10759.0,"(10001, 80071)"
4,RLT_10,2024,10001,FIN,8002,12023.0,50001,PROD,8002.0,12023.0,90005,ADD,NaN,585.0,"(10001, 90005)"


In [9]:
first_level_expanded.loc[first_level_expanded['fin_material_id'] == 10000, 'component_id'].unique()

array([80070, 90000, 90001, 80010, 90002, 90003, 80000, 90004, 70000,
       90005])

In [10]:
first_level_expanded.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 15 columns):
 #   Column                              Non-Null Count  Dtype  
---  ------                              --------------  -----  
 0   plant_id                            100 non-null    object 
 1   year                                100 non-null    int64  
 2   fin_material_id                     100 non-null    int64  
 3   fin_material_release_type           100 non-null    object 
 4   fin_material_production_type        100 non-null    int64  
 5   fin_production_quantity             100 non-null    float64
 6   prod_material_id                    100 non-null    int64  
 7   prod_material_release_type          100 non-null    object 
 8   prod_material_production_type       100 non-null    float64
 9   prod_production_quantity            100 non-null    float64
 10  component_id                        100 non-null    int64  
 11  component_material_release_type     100 non-nu

In [15]:
out = first_level_expanded.rename(columns={'plant_id': 'plant'}).copy()

out['path_str'] = out['path'].apply(
    lambda p: '/'.join(map(str, p)) if isinstance(p, (list, tuple)) else str(p)
)

sort_keys = ['plant', 'year', 'fin_material_id', 'path_str', 'prod_material_id', 'component_id']
out_ordered = out.sort_values(by=sort_keys).reset_index(drop=True)

final_columns = [
    'plant',
    'year',
    'fin_material_id',
    'fin_material_release_type',
    'fin_material_production_type',
    'fin_production_quantity',
    'prod_material_id',
    'prod_material_release_type',
    'prod_material_production_type',
    'prod_production_quantity',
    'component_id',
    'component_material_release_type',
    'component_material_production_type',
    'component_consumption_quantity'
 ]
out_ordered = out_ordered[final_columns]

out_path = './data/bom_exploded_result_pandas.csv'
out_ordered.to_csv(out_path, index=False)

print(f"Результат сохранен в {out_path}")
print(f"Строк: {len(out_ordered)}")

Результат сохранен в ./data/bom_exploded_result_pandas.csv
Строк: 100
